<a href="https://colab.research.google.com/github/charmy-patel/practicals/blob/bigdata/GraphX_FB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install graphframes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 2.6 MB/s eta 0:00:00


In [3]:
!wget https://snap.stanford.edu/data/facebook_combined.txt.gz
!gunzip facebook_combined.txt.gz

--2025-09-17 05:22:30--  https://snap.stanford.edu/data/facebook_combined.txt.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 218576 (213K) [application/x-gzip]
Saving to: ‘facebook_combined.txt.gz’

facebook_combined.t 100%[===================>] 213.45K   223KB/s    in 1.0s    

2025-09-17 05:22:31 (223 KB/s) - ‘facebook_combined.txt.gz’ saved [218576/218576]



In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GraphFramesExample") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.2-spark3.2-s_2.12") \
    .getOrCreate()


In [5]:
# Load edges (friendship pairs)
edges = spark.read.csv("facebook_combined.txt",
                       sep=" ", header=False, inferSchema=True).toDF("src", "dst")

# Build vertices (unique users)
#GraphFrames requires vertices table.
#We take all unique users (both from src and dst) → each becomes a node.
vertices = edges.selectExpr("src as id").union(edges.selectExpr("dst as id")).distinct()

print("Total Vertices:", vertices.count())
print("Total Edges:", edges.count())

Total Vertices: 4039
Total Edges: 88234


In [8]:
from graphframes import GraphFrame
from pyspark.sql.functions import col
# Force all ids to string
vertices = vertices.withColumn("id", col("id").cast("string"))
edges = edges.withColumn("src", col("src").cast("string")) \
             .withColumn("dst", col("dst").cast("string"))


g = GraphFrame(vertices, edges)

print("Total Vertices:", g.vertices.count())
print("Total Edges:", g.edges.count())

# Get SparkContext from SparkSession
sc = spark.sparkContext

# Set checkpoint directory
# Graph algorithms (like Connected Components) create very long computation chains.
#Checkpointing cuts those chains by saving intermediate results to disk.
#Without this, GraphFrames throws the Checkpoint directory is not set error.
sc.setCheckpointDir("/content/drive/My Drive/Colab Notebooks/checkpoints")

#components = g.connectedComponents()
#components.show(10, truncate=False)

g.vertices.printSchema()
g.shortestPaths(landmarks=["0", "100", "200"]).show(10, truncate=False)

Total Vertices: 4039
Total Edges: 88234
root
 |-- id: string (nullable = true)

+----+----------+
|id  |distances |
+----+----------+
|148 |{200 -> 2}|
|463 |{}        |
|471 |{}        |
|496 |{}        |
|1088|{}        |
|1238|{}        |
|1342|{}        |
|1580|{}        |
|1591|{}        |
|1645|{}        |
+----+----------+
only showing top 10 rows



PageRank (to find influential users)

In [ ]:
results = g.pageRank(resetProbability=0.15, maxIter=10)
results.vertices.orderBy("pagerank", ascending=False).show(25, truncate=False)

/usr/local/lib/python3.12/dist-packages/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+----+------------------+
|id  |pagerank          |
+----+------------------+
|1911|40.17302003518893 |
|3434|38.2119696897726  |
|2655|37.63024755334723 |
|1902|37.243511110380986|
|1888|28.028255649535474|
|2649|25.511310917057614|
|1907|21.24097102153389 |
|3971|20.558757839388296|
|2654|20.200635873097053|
|1910|17.551329484911335|
|1894|16.221770472762415|
|1898|15.312986994573006|
|1882|15.298854320152195|
|3430|15.249132452404812|
|3426|14.685238970869753|
|2660|14.590672282147839|
|2642|12.43879608134838 |
|1891|12.017851816789964|
|3422|11.960462389548606|
|332 |11.87872611754509 |
|2653|11.232361289293443|
|3968|11.122514968244932|
|1897|10.914439340307116|
|1906|10.755647149777781|
|1879|10.602986360197612|
+----+------------------+
only showing top 25 rows



Connected Components (find groups/communities)

In [9]:
# Get SparkContext from SparkSession
sc = spark.sparkContext

# Set checkpoint directory
# Graph algorithms (like Connected Components) create very long computation chains.
#Checkpointing cuts those chains by saving intermediate results to disk.
#Without this, GraphFrames throws the Checkpoint directory is not set error.
sc.setCheckpointDir("/content/drive/My Drive/Colab Notebooks/checkpoints")

#Finds groups of users where each user is reachable from any other in the group.
#In social networks → these are clusters of friends.
#g.connectedComponents().show(10, truncate=False)

components = g.connectedComponents()
components.select("component").distinct().count()


1

In [ ]:
communities = g.labelPropagation(maxIter=5)
communities.show(10, truncate=False)

+----+-----+
|id  |label|
+----+-----+
|3558|3591 |
|1084|1205 |
|3702|3672 |
|3007|3277 |
|667 |559  |
|1053|107  |
|1894|1824 |
|2493|2344 |
|1325|1187 |
|3517|3968 |
+----+-----+
only showing top 10 rows



In [ ]:

g.shortestPaths(landmarks=[0, 100, 200]).show(10, truncate=False)


+----+---------+
|id  |distances|
+----+---------+
|3558|{}       |
|1084|{}       |
|3702|{}       |
|3007|{}       |
|667 |{}       |
|1053|{}       |
|1894|{}       |
|2493|{}       |
|1325|{}       |
|3517|{}       |
+----+---------+
only showing top 10 rows

